In [6]:
### Dependencies
# Base Dependencies
import os
import pickle
import sys

# LinAlg / Stats / Plotting Dependencies
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# Torch Dependencies
import torch
import torch.multiprocessing
import torchvision
import torch.utils.data.dataset as Dataset
from torchvision import transforms

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

torch.multiprocessing.set_sharing_strategy("file_system")

# Model Architectures
from models.vision_transformer import vit_tiny


In [7]:
### Functions for Loading + Saving + Visualizing Patch Embeddings
def save_embeddings(
    model,
    fname,
    dataloader,
    dataset=None,
    is_imagefolder=False,
    save_patches=False,
    sprite_dim=128,
    overwrite=False,
):

    if os.path.isfile("%s.pkl" % fname) and (overwrite == False):
        return None

    embeddings, labels = [], []
    patches = []

    # TODO, che è sta sprite_dim?

    for batch, target in tqdm(dataloader):
        if save_patches:
            for img in batch:
                patches.append(tensor2im(input_image=img).resize(sprite_dim))

        with torch.no_grad():
            batch = batch.to(device)
            embeddings.append(model(batch).detach().cpu().numpy())
            labels.append(target.numpy())

    embeddings = np.vstack(embeddings)
    labels = np.vstack(labels).squeeze()

    if is_imagefolder:
        id2label = dict(map(reversed, dataset.class_to_idx.items()))
        labels = np.array(list(map(id2label.get, labels.ravel())))

    asset_dict = {"embeddings": embeddings, "labels": labels}
    if save_patches:
        asset_dict.update({"patches": patches})
    with open("%s.pkl" % (fname), "wb") as handle:
        pickle.dump(asset_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [8]:
### Helper Functions for Normalization + Loading in pytorch_lightning SSL encoder (for SimCLR)
def eval_transforms(pretrained=False):
    # TODO need to understand if this is useful or not
    if pretrained:
        mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    else:
        mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
    trnsfrms_val = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=mean, std=std)]
    )
    return trnsfrms_val


def create_embeddings(
    embeddings_dir,
    enc_name,
    save_patches=False,
    sprite_dim=128,
    patch_datasets="path/to/patch/datasets",
    assets_dir="./ckpts/",
    disentangle=-1,
    stage=-1,
):
    print("Extracting Features for '%s' via '%s'" % (patch_datasets, enc_name))
    if "dino" in enc_name:
            ckpt_path = os.path.join(assets_dir, enc_name + ".pth")
            assert os.path.isfile(ckpt_path)
            model = vit_tiny(patch_size=16)
            state_dict = torch.load(ckpt_path, map_location="cpu")["teacher"]
            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
            state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
            missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
            print("Missing Keys:", missing_keys)
            print("Unexpected Keys:", unexpected_keys)
            eval_t = eval_transforms(pretrained=False)
    else:
        pass

    model = model.to(device)
    model.eval()

    if "dino" in enc_name:
        _model = model
        if stage == -1:
            model = _model
        else:
            model = lambda x: torch.cat(
                [x[:, 0] for x in _model.get_intermediate_layers(x, stage)], dim=-1
            )

    if stage != -1:
        _stage = "_s%d" % stage
    else:
        _stage = ""

    ### Train
    dataroot = patch_datasets
    # image folder richiede che le immagini siano divise in sotto directory in cui le immagini appartengono ad una data classe
    dataset = torchvision.datasets.ImageFolder(dataroot, transform=eval_t)

    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=4
    )
    fname = os.path.join(embeddings_dir, "%s%s" % (enc_name, _stage))
    save_embeddings(
        model=model,
        fname=fname,
        dataloader=dataloader,
        dataset=dataset,
        save_patches=save_patches,
        sprite_dim=sprite_dim,
        is_imagefolder=True,
        overwrite=True
    )

In [ ]:
# Update this path to point to your local patches directory
patch_datasets = "./data/patches/"
library_path = "./embeddings_patch_library/"
os.makedirs(library_path, exist_ok=True)
model = "dino"

create_embeddings(
    patch_datasets=patch_datasets, embeddings_dir=library_path, enc_name=model
)